In [1]:
import pandas as pd
import numpy as np

In [2]:
df_credit = pd.read_parquet("data/ready_to_train/df_credit_final_21_feats_20250702.parquet")

In [3]:
all_feats = [col for col in df_credit.columns if col not in ["Transaction Datetime","Confirmed"]]
category_feats = ["CustomerSex","MCCCategory"]
numerical_feats = [col for col in all_feats if col not in category_feats]

In [4]:
for col in category_feats:
    df_credit[col] = df_credit[col].fillna("__missing__")

for col in numerical_feats:
    df_credit[col] = df_credit[col].fillna(0)

In [5]:
df_train_template = pd.read_csv("data/others/0_Training Template.csv")
df_validation_template = pd.read_csv("data/others/0_Validation Template.csv")

In [6]:
rename_col_map = {
    'Transaction Serial No': 'Identifier',
    'Confirmed': 'Fraud_Clean',
    'AvgTimeFirstTxnToCurrentMCCL30D': 'Transaction_Summary_Calculations_Fraud.AvgTimeFirstTxnToCurrentMCCL30D.1',
    'AvgTimeFirstTxnToCurrentMCCL15min': 'Transaction_Summary_Calculations_Fraud.AvgTimeFirstTxnToCurrentMCCL15min.1',
    'CntUniqueCardNoByCurrencyCodeL30D': 'Transaction_Summary_Calculations_Fraud.CntUniqueCardNoByCurrencyCodeL30D.1',
    'RatioCntUniqueCardNoByCurrencyCodeL30DL15min': 'Transaction_Summary_Calculations_Fraud.RatioCntUniqueCardNoByCurrencyCodeL30DL15min.1',
    'CntUniqueMCCByCardNoL15min': 'Transaction_Summary_Calculations_Fraud.CntUniqueMCCByCardNoL15min.1',
    'RatioCntUniqueMCCByCardNoL30DL15min': 'Transaction_Summary_Calculations_Fraud.RatioCntUniqueMCCByCardNoL30DL15min.1',
    'CustomerSex': 'Transaction_Summary_Calculations_Fraud.CustomerSex.1',
    'MCCCategory': 'Transaction_Summary_Calculations_Fraud.MCCCategory.1',
    'TransactionAmount': 'Transaction_Summary_Fraud.Transaction_Amount.1',
    'TotalTrxAmount10Mi': 'Transaction_Summary_Calculations_Fraud.TotalTrxAmount10Mi.1',
    'IsTop10HighRiskMCCLast30D': 'Transaction_Summary_Calculations_Fraud.IsTop10HighRiskMCCLast30D.1',
    'MaxAmtToMCCL30D': 'Transaction_Summary_Calculations_Fraud.MaxAmtToMCCL30D.1',
    'RatioSumAmtToMCCL30DL15min': 'Transaction_Summary_Calculations_Fraud.RatioSumAmtToMCCL30DL15min.1',
    'RatioTxnCountToMCCL30DL15min': 'Transaction_Summary_Calculations_Fraud.RatioTxnCountToMCCL30DL15min.1',
    'TxnCountL15min': 'Transaction_Summary_Calculations_Fraud.TxnCountL15min.1',
    'TxnCountToCountryCodeL15min': 'Transaction_Summary_Calculations_Fraud.TxnCountToCountryCodeL15min.1',
    'RatioTxnCountL30DL15min': 'Transaction_Summary_Calculations_Fraud.RatioTxnCountL30DL15min.1',
    'RatioTxnCountToCountryCodeL30DL15min': 'Transaction_Summary_Calculations_Fraud.RatioTxnCountToCountryCodeL30DL15min.1',
    'TxnCountToPOSModeL15min': 'Transaction_Summary_Calculations_Fraud.TxnCountToPOSModeL15min.1',
    'RatioTxnCountToPOSModeL30DL15min': 'Transaction_Summary_Calculations_Fraud.RatioTxnCountToPOSModeL30DL15min.1',
    'TxnTimeDifference': 'Transaction_Summary_Calculations_Fraud.TxnTimeDifference.1'
}

In [7]:
df_credit_final = df_credit.reset_index().rename(columns=rename_col_map)

In [10]:
df_credit_final["Fraud_Clean"] = df_credit_final["Fraud_Clean"].astype(int)

In [13]:
from src.model_pipeline import ModelPipeline

pipeline = ModelPipeline()

split_date = '2025-05-01'
df_splits = pipeline.split_data_by_date(
    df=df_credit_final,
    date_column="Transaction Datetime",
    split_date=split_date
)

Initializing ModelPipeline with model type: xgboost
Retrieving model instance for type: xgboost
Splitting data by date (pre-preprocessing)...
Train samples: 1197282, Test samples: 201546
Data splitting complete.


In [14]:
df_splits["df_train"] = df_splits["df_train"].drop("Transaction Datetime", axis=1)
df_splits["df_train"]["Fraud_Clean"] = df_splits["df_train"]["Fraud_Clean"].astype(int)
df_splits["df_train"][df_train_template.columns.to_list()].to_csv("data/ready_to_train/TRAIN_CREDIT_df_sit__int_label_20250710.csv", index=False)

df_splits["df_test"] = df_splits["df_test"].drop("Transaction Datetime", axis=1)
df_splits["df_test"]["Fraud_Clean"] = df_splits["df_test"]["Fraud_Clean"].astype(int)
df_splits["df_test"][df_validation_template.columns.to_list()].to_csv("data/ready_to_train/VALIDATION_CREDIT_df_sit__int_label_20250710.csv", index=False)